# Notebook 03: Forecasting Models & Validation

**Dataset**: FreshRetailNet-50K  
**Goal**: Learn how to properly train and validate demand forecasting models

---

In this notebook we cover:

1. **Setup & Data Preparation** -- Load data, engineer features, prepare for modeling
2. **The Problem with Single Splits** -- Why naive train/val splits are fragile
3. **Temporal Cross-Validation** -- The right way to evaluate time-series models
4. **Feature Importance** -- What drives our forecasts?
5. **Model Comparison** -- MAE vs Huber vs Quantile vs Ensemble
6. **Key Takeaways** -- Summary of lessons learned

This notebook is **standalone** -- all feature engineering code is included here rather than imported from the pipeline.

---
## Section 1: Setup & Data Preparation

We load the FreshRetailNet training data, sample 3000 store-product (SP) combinations stratified by stockout rate, recover censored demand, and build a full feature engineering pipeline (~120 features).

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import gc

import os
NB_OUT = os.path.join('..', 'notebook_output')
os.makedirs(NB_OUT, exist_ok=True)

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_style('whitegrid')

print("Libraries loaded.")

### 1.1 Load & Sample Data

We use stratified sampling by stockout rate so our sample represents the full spectrum of demand patterns -- from products that rarely stock out to those that frequently do.

In [ ]:
N_SP = 3000

TRAIN_PATH = '../../data/freshretailnet/raw/data/train.parquet'
EVAL_PATH = '../../data/freshretailnet/raw/data/eval.parquet'

cols = [
    'city_id', 'store_id', 'management_group_id', 'first_category_id',
    'second_category_id', 'third_category_id', 'product_id', 'dt',
    'sale_amount', 'stock_hour6_22_cnt', 'discount', 'holiday_flag',
    'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'
]

train = pd.read_parquet(TRAIN_PATH, columns=cols)
train['sp'] = train['store_id'] * 10000 + train['product_id']

# Stratified sampling by stockout rate
so = train.groupby('sp')['stock_hour6_22_cnt'].apply(lambda x: (x > 0).mean())
so = so.reset_index()
so.columns = ['sp', 'sor']
so['bin'] = pd.qcut(so['sor'], q=5, labels=False, duplicates='drop')

sampled = so.groupby('bin').apply(
    lambda x: x.sample(min(len(x), N_SP // 5), random_state=42)
).reset_index(drop=True)['sp'].values

if len(sampled) < N_SP:
    extra = np.random.choice(
        list(set(so['sp']) - set(sampled)),
        N_SP - len(sampled), replace=False
    )
    sampled = np.concatenate([sampled, extra])
sampled = set(sampled[:N_SP])

train = train[train['sp'].isin(sampled)].copy()
train['dt'] = pd.to_datetime(train['dt'])

# Downcast for memory efficiency
for c in train.select_dtypes('int64').columns:
    if c != 'sp':
        train[c] = train[c].astype('int32')
for c in train.select_dtypes('float64').columns:
    train[c] = train[c].astype('float32')

print(f"Train shape: {train.shape}")
print(f"Unique SPs:  {train['sp'].nunique()}")
print(f"Date range:  {train['dt'].min().date()} to {train['dt'].max().date()}")
print(f"Stockout rate: {(train['stock_hour6_22_cnt'] > 0).mean() * 100:.1f}%")
print(f"Memory: {train.memory_usage(deep=True).sum() / 1e6:.0f} MB")

del so
gc.collect()

### 1.2 Censored Demand Recovery

When a product stocks out, observed sales undercount true demand. We use the **proportional method** to estimate what demand *would have been* if stock were available:

- **Partial stockout** (some hours with no stock): Scale up observed sales by `operating_hours / available_hours`
- **Full stockout** (entire day): Use 14-day rolling average of non-stockout days
- **Correction**: Add a small adjustment based on the product's demand variability

The recovered demand (`dem_rec`) is used as a **feature**, not as the target. We still predict `sale_amount`.

In [ ]:
OP = 16  # operating hours (6am-10pm)

train = train.sort_values(['sp', 'dt']).reset_index(drop=True)
train['cens'] = (train['stock_hour6_22_cnt'] > 0).astype('int8')
train['so_frac'] = (train['stock_hour6_22_cnt'] / OP).astype('float32')
train['dem_rec'] = train['sale_amount'].values.copy().astype(np.float64)

partial = (train['stock_hour6_22_cnt'] > 0) & (train['stock_hour6_22_cnt'] < OP)
full = train['stock_hour6_22_cnt'] >= OP

# Partial stockout: proportional scaling
avail = (OP - train['stock_hour6_22_cnt']).clip(lower=1)
train.loc[partial, 'dem_rec'] = (
    train.loc[partial, 'sale_amount'] * (OP / avail[partial])
)

# Full stockout: rolling average of non-stockout days
no_so = (train['cens'] == 0).astype(float)
train['_s'] = train['sale_amount'] * no_so
rs = train.groupby('sp')['_s'].transform(lambda x: x.rolling(14, min_periods=1).sum())
rc = train.groupby('sp').apply(
    lambda g: no_so.loc[g.index].rolling(14, min_periods=1).sum()
).reset_index(level=0, drop=True).sort_index()
avg_ns = rs / rc.clip(lower=1)
train.loc[full, 'dem_rec'] = avg_ns[full]
train.drop('_s', axis=1, inplace=True)

# Simple correction factor
sp_std = train.groupby('sp')['sale_amount'].transform('std').fillna(0)
correction = sp_std * train['so_frac'] * 0.3
train.loc[train['cens'] == 1, 'dem_rec'] += correction[train['cens'] == 1]

train['dem_rec'] = train['dem_rec'].clip(lower=0).astype('float32')

print(f"Censored observations: {train['cens'].sum():,} ({train['cens'].mean() * 100:.1f}%)")
print(f"Sale mean:      {train['sale_amount'].mean():.4f}")
print(f"Recovered mean: {train['dem_rec'].mean():.4f}")
print(f"Recovery uplift: {(train['dem_rec'].mean() / train['sale_amount'].mean() - 1) * 100:.1f}%")

gc.collect()

### 1.3 Feature Engineering

We build ~120 features organized into several groups:

| Group | Examples | Count |
|-------|---------|-------|
| Temporal | day-of-week, month, cyclical sin/cos | ~12 |
| Lags | 1, 2, 3, 5, 7, 14, 21, 28 day lags | ~16 |
| Rolling stats | 3/7/14/28-day mean, std, min, max, median | ~30 |
| EWMA | 7/14-day exponential weighted moving avg | ~4 |
| Stockout features | yesterday's stockout, 7/14-day SO rate | ~6 |
| Variability & trend | CV, trend ratios, momentum | ~8 |
| Cross features | discount x holiday, temp x precip, etc. | ~7 |
| Global stats | SP/product/store/city/category means | ~10 |
| Hierarchy features | store volume, category-level, clustering | ~15 |

In [ ]:
N_CLUSTERS = 25

def make_features(df, cluster_model=None):
    """Build ~120 features including hierarchy exploitation and clustering."""
    df = df.sort_values(['sp', 'dt']).reset_index(drop=True)
    g = df.groupby('sp')
    targets = {'s': 'sale_amount', 'r': 'dem_rec'}

    # ---- Temporal features ----
    df['dow'] = df['dt'].dt.dayofweek.astype('int8')
    df['dom'] = df['dt'].dt.day.astype('int8')
    df['woy'] = df['dt'].dt.isocalendar().week.astype('int8')
    df['month'] = df['dt'].dt.month.astype('int8')
    df['wknd'] = (df['dow'] >= 5).astype('int8')
    df['doy'] = df['dt'].dt.dayofyear.astype('int16')
    for cyc, period in [('dow', 7), ('dom', 31), ('woy', 52)]:
        df[f'{cyc}_sin'] = np.sin(2 * np.pi * df[cyc] / period).astype('float32')
        df[f'{cyc}_cos'] = np.cos(2 * np.pi * df[cyc] / period).astype('float32')

    # ---- Lag features ----
    for pfx, col in targets.items():
        for lag in [1, 2, 3, 5, 7, 14, 21, 28]:
            df[f'{pfx}_l{lag}'] = g[col].shift(lag).astype('float32')

    # Diff / momentum
    df['s_d1'] = (df['s_l1'] - df['s_l2']).astype('float32')
    df['s_d7'] = (df['s_l1'] - df['s_l7']).astype('float32')
    df['r_d1'] = (df['r_l1'] - g['dem_rec'].shift(2)).astype('float32')

    # ---- Rolling statistics ----
    shifted_s = g['sale_amount'].shift(1)
    shifted_r = g['dem_rec'].shift(1)

    for pfx, shifted in [('s', shifted_s), ('r', shifted_r)]:
        for w in [3, 7, 14, 28]:
            r = shifted.groupby(df['sp']).rolling(w, min_periods=1)
            df[f'{pfx}_m{w}'] = r.mean().reset_index(level=0, drop=True).astype('float32')
            df[f'{pfx}_sd{w}'] = r.std().reset_index(level=0, drop=True).astype('float32')
            if w in [7, 28]:
                df[f'{pfx}_mx{w}'] = r.max().reset_index(level=0, drop=True).astype('float32')
                df[f'{pfx}_mn{w}'] = r.min().reset_index(level=0, drop=True).astype('float32')
                df[f'{pfx}_md{w}'] = r.median().reset_index(level=0, drop=True).astype('float32')
            gc.collect()

    # Quantile features
    for w in [14, 28]:
        r = shifted_s.groupby(df['sp']).rolling(w, min_periods=1)
        df[f's_q25_{w}'] = r.quantile(0.25).reset_index(level=0, drop=True).astype('float32')
        df[f's_q75_{w}'] = r.quantile(0.75).reset_index(level=0, drop=True).astype('float32')

    # ---- EWMA ----
    for pfx, shifted in [('s', shifted_s), ('r', shifted_r)]:
        for s in [7, 14]:
            df[f'{pfx}_ew{s}'] = shifted.groupby(df['sp']).transform(
                lambda x: x.ewm(span=s, min_periods=1).mean()
            ).astype('float32')

    del shifted_s, shifted_r
    gc.collect()

    # ---- Stockout features ----
    cs = g['cens'].shift(1)
    df['so1'] = cs.astype('float32')
    df['so7'] = g['cens'].shift(7).astype('float32')
    for w in [7, 14]:
        df[f'sor{w}'] = cs.groupby(df['sp']).rolling(w, min_periods=1).mean() \
            .reset_index(level=0, drop=True).astype('float32')
    hs = g['stock_hour6_22_cnt'].shift(1)
    for w in [7, 14]:
        df[f'soh{w}'] = hs.groupby(df['sp']).rolling(w, min_periods=1).mean() \
            .reset_index(level=0, drop=True).astype('float32')
    del cs, hs
    gc.collect()

    # ---- Variability & trend ----
    df['cv7'] = (df['s_sd7'] / df['s_m7'].clip(lower=0.01)).astype('float32')
    df['cv28'] = (df['s_sd28'] / df['s_m28'].clip(lower=0.01)).astype('float32')
    df['tr_7_28'] = (df['s_m7'] - df['s_m28']).astype('float32')
    df['tr_3_14'] = (df['s_m3'] - df['s_m14']).astype('float32')

    # Ratio features
    df['l1_m7'] = (df['s_l1'] / df['s_m7'].clip(lower=0.01)).astype('float32')
    df['m7_m28'] = (df['s_m7'] / df['s_m28'].clip(lower=0.01)).astype('float32')
    df['rec_obs_ratio'] = (df['r_m7'] / df['s_m7'].clip(lower=0.01)).astype('float32')

    # ---- DOW profile ----
    df['dow_prof'] = df.groupby(['sp', 'dow'])['sale_amount'].transform('mean').astype('float32')
    df['dow_prof_r'] = df.groupby(['sp', 'dow'])['dem_rec'].transform('mean').astype('float32')

    # ---- Cross features ----
    df['d_h'] = (df['discount'] * df['holiday_flag']).astype('float32')
    df['d_a'] = (df['discount'] * df['activity_flag']).astype('float32')
    df['t_p'] = (df['avg_temperature'] * df['precpt']).astype('float32')
    df['w_h'] = (df['wknd'] * df['holiday_flag']).astype('int8')
    df['h_a'] = (df['holiday_flag'] * df['activity_flag']).astype('int8')
    df['temp_dev'] = (df['avg_temperature'] - df.groupby('sp')['avg_temperature'].transform('mean')).astype('float32')
    df['hum_dev'] = (df['avg_humidity'] - df.groupby('sp')['avg_humidity'].transform('mean')).astype('float32')

    # ---- Global stats ----
    for grp, pfx in [('sp', 'sp'), ('product_id', 'pd'), ('store_id', 'st'), ('city_id', 'ct')]:
        df[f'{pfx}_m'] = df.groupby(grp)['sale_amount'].transform('mean').astype('float32')
        df[f'{pfx}_s'] = df.groupby(grp)['sale_amount'].transform('std').fillna(0).astype('float32')
    df['cat1_m'] = df.groupby('first_category_id')['sale_amount'].transform('mean').astype('float32')
    df['cat2_m'] = df.groupby('second_category_id')['sale_amount'].transform('mean').astype('float32')

    # ---- Hierarchy features ----
    df['cat3_m'] = df.groupby('third_category_id')['sale_amount'].transform('mean').astype('float32')
    df['cat3_s'] = df.groupby('third_category_id')['sale_amount'].transform('std').fillna(0).astype('float32')

    df['sp_vs_cat2'] = (df['sp_m'] / df['cat2_m'].clip(lower=0.01)).astype('float32')
    df['sp_vs_cat3'] = (df['sp_m'] / df['cat3_m'].clip(lower=0.01)).astype('float32')
    df['sp_vs_store'] = (df['sp_m'] / df['st_m'].clip(lower=0.01)).astype('float32')

    df['store_daily_vol'] = df.groupby(['store_id', 'dt'])['sale_amount'].transform('sum').astype('float32')
    df['store_so_rate'] = df.groupby('store_id')['cens'].transform('mean').astype('float32')

    cat2_daily = df.groupby(['second_category_id', 'dt'])['sale_amount'].transform('mean')
    df['cat2_daily_m'] = cat2_daily.astype('float32')

    # ---- Store-product clustering ----
    sp_behavior = df.groupby('sp').agg(
        mean_demand=('sale_amount', 'mean'),
        std_demand=('sale_amount', 'std'),
        so_rate=('cens', 'mean'),
        zero_rate=('sale_amount', lambda x: (x == 0).mean()),
    ).fillna(0)
    sp_behavior['cv'] = sp_behavior['std_demand'] / sp_behavior['mean_demand'].clip(lower=0.01)

    df['_is_wknd'] = (df['dow'] >= 5).astype(float)
    wknd_mean = df[df['_is_wknd'] == 1].groupby('sp')['sale_amount'].mean()
    wkday_mean = df[df['_is_wknd'] == 0].groupby('sp')['sale_amount'].mean()
    sp_behavior['wknd_ratio'] = (wknd_mean / wkday_mean.clip(lower=0.01)).fillna(1.0)
    df.drop('_is_wknd', axis=1, inplace=True)

    scaler = StandardScaler()
    sp_features_scaled = scaler.fit_transform(sp_behavior.values)

    if cluster_model is None:
        cluster_model = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
        cluster_model.fit(sp_features_scaled)

    sp_behavior['cluster_id'] = cluster_model.labels_
    sp_cluster_map = sp_behavior['cluster_id'].to_dict()
    df['cluster_id'] = df['sp'].map(sp_cluster_map).fillna(0).astype('int8')

    df['cluster_m'] = df.groupby('cluster_id')['sale_amount'].transform('mean').astype('float32')
    df['cluster_s'] = df.groupby('cluster_id')['sale_amount'].transform('std').fillna(0).astype('float32')

    # ---- Finalize ----
    df = df.fillna(0)
    gc.collect()
    return df, cluster_model


def get_fcols(df):
    """Get feature columns (everything except identifiers, targets, and metadata)."""
    excl = {'dt', 'sale_amount', 'dem_rec', 'sp', 'cens', 'so_frac'}
    return [c for c in df.columns if c not in excl]


print("Building features (this takes a minute)...")
train, cluster_model = make_features(train)
fc = get_fcols(train)

print(f"\nFeature count: {len(fc)}")
print(f"Dataset shape: {train.shape}")
print(f"Memory: {train.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"\nSample features: {fc[:10]}")
print(f"...")
print(f"Last features:  {fc[-10:]}")

---
## Section 2: The Problem with Single Splits

The most common approach to model validation is a **single temporal split**: use the last N days as a hold-out validation set and train on everything before it. This is better than random splitting (which would leak future information), but it has a critical weakness.

Let's see why.

### 2.1 The Naive Approach: Last 7 Days as Validation

We take the last 7 days of training data as our validation set and train a LightGBM model with MAE loss on the rest.

In [ ]:
# LightGBM base parameters (used throughout the notebook)
LGB_BASE = {
    'boosting_type': 'gbdt',
    'num_leaves': 127,
    'learning_rate': 0.05,
    'feature_fraction': 0.75,
    'bagging_fraction': 0.75,
    'bagging_freq': 5,
    'min_child_samples': 30,
    'lambda_l1': 0.1,
    'lambda_l2': 1.0,
    'max_depth': -1,
    'min_gain_to_split': 0.01,
    'verbose': -1,
    'n_jobs': -1,
    'seed': 42,
}

TARGET = 'sale_amount'
dates = sorted(train['dt'].unique())
warmup_date = dates[27]  # first 28 days for feature warmup

# Single split: last 7 days as validation
split_date = dates[-8]  # train up to this date
val_start = dates[-7]
val_end = dates[-1]

tr_mask = (train['dt'] > warmup_date) & (train['dt'] <= split_date)
vl_mask = (train['dt'] >= val_start) & (train['dt'] <= val_end)

Xtr, ytr = train.loc[tr_mask, fc].values, train.loc[tr_mask, TARGET].values
Xvl, yvl = train.loc[vl_mask, fc].values, train.loc[vl_mask, TARGET].values

# Weights: downweight censored observations
wtr = np.ones(len(ytr), dtype='float32')
wtr[train.loc[tr_mask, 'cens'].values == 1] = 0.5

print(f"Training period:   {warmup_date.date()} to {split_date.date()} ({tr_mask.sum():,} rows)")
print(f"Validation period: {val_start.date()} to {val_end.date()} ({vl_mask.sum():,} rows)")

In [ ]:
# Train a single MAE model on this split
params = {**LGB_BASE, 'objective': 'regression_l1', 'metric': 'mae'}
dtrain = lgb.Dataset(Xtr, ytr, weight=wtr, feature_name=fc)
dval = lgb.Dataset(Xvl, yvl, feature_name=fc, reference=dtrain)

single_model = lgb.train(
    params, dtrain, 800,
    valid_sets=[dval], valid_names=['val'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

single_pred = np.clip(single_model.predict(Xvl), 0, None)
single_mae = mean_absolute_error(yvl, single_pred)
single_rmse = np.sqrt(mean_squared_error(yvl, single_pred))
single_corr = np.corrcoef(yvl, single_pred)[0, 1]

print(f"\n--- Single Split Results ---")
print(f"MAE:  {single_mae:.4f}")
print(f"RMSE: {single_rmse:.4f}")
print(f"Corr: {single_corr:.4f}")
print(f"Best iteration: {single_model.best_iteration}")

del Xtr, ytr, Xvl, yvl, wtr, dtrain, dval
gc.collect()

### 2.2 Why Is This Fragile?

The single-split result looks fine, but consider this: **what if that particular week was unusually easy or hard to predict?**

Demand varies week-to-week due to:
- **Holidays** -- a holiday week might show unusual patterns
- **Weather** -- a heat wave or storm changes buying behavior
- **Promotions** -- an activity week can spike demand unpredictably
- **Random variation** -- some weeks just happen to be more predictable

A single validation window gives you **one number**. You have no idea how confident to be in it. Let's demonstrate this problem.

In [ ]:
# Try 3 different 7-day validation windows and see how results change
available_dates = [d for d in dates if d > warmup_date]
n_avail = len(available_dates)

windows = [
    ('Window A (earliest)',  available_dates[7:14]),
    ('Window B (middle)',    available_dates[n_avail//2 : n_avail//2 + 7]),
    ('Window C (latest)',    available_dates[-7:]),
]

window_results = []

for name, val_dates in windows:
    val_start_d = val_dates[0]
    val_end_d = val_dates[-1]
    train_end_d = available_dates[available_dates.index(val_start_d) - 1]

    tr_m = (train['dt'] > warmup_date) & (train['dt'] <= train_end_d)
    vl_m = train['dt'].isin(val_dates)

    Xt, yt = train.loc[tr_m, fc].values, train.loc[tr_m, TARGET].values
    Xv, yv = train.loc[vl_m, fc].values, train.loc[vl_m, TARGET].values

    wt = np.ones(len(yt), dtype='float32')
    wt[train.loc[tr_m, 'cens'].values == 1] = 0.5

    p = {**LGB_BASE, 'objective': 'regression_l1', 'metric': 'mae'}
    dt_ = lgb.Dataset(Xt, yt, weight=wt, feature_name=fc)
    dv_ = lgb.Dataset(Xv, yv, feature_name=fc, reference=dt_)

    mdl = lgb.train(
        p, dt_, 800,
        valid_sets=[dv_], valid_names=['val'],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )

    pred = np.clip(mdl.predict(Xv), 0, None)
    mae_val = mean_absolute_error(yv, pred)

    window_results.append({
        'Window': name,
        'Val period': f"{val_start_d.date()} to {val_end_d.date()}",
        'Train size': tr_m.sum(),
        'Val size': vl_m.sum(),
        'MAE': mae_val,
    })

    del Xt, yt, Xv, yv, wt, dt_, dv_, mdl
    gc.collect()

wr_df = pd.DataFrame(window_results)
print("\n" + "=" * 70)
print("  SAME MODEL, DIFFERENT VALIDATION WINDOWS")
print("=" * 70)
print(wr_df.to_string(index=False))

mae_range = wr_df['MAE'].max() - wr_df['MAE'].min()
mae_mean = wr_df['MAE'].mean()
print(f"\nMAE range across windows: {mae_range:.4f}")
print(f"MAE varies by {mae_range / mae_mean * 100:.1f}% depending on which week you hold out!")
print("\nThis is why a single split is dangerous -- your result is fragile.")

### Lesson from Section 2

> **A single temporal split gives you a point estimate with no uncertainty band.** You do not know if your MAE of X is because your model is genuinely good, or because you happened to pick an easy week. If a colleague trains on a different split and gets a different number, neither of you can tell who is right.

The solution? **Temporal cross-validation.**

---
## Section 3: Temporal Cross-Validation

### 3.1 Expanding Window vs Rolling Window

There are two main approaches to temporal CV:

**Expanding Window** (what we use): The training set grows with each fold.

```
                     Warmup |  Training Data       | Validation
Fold 1:  [====WARMUP====]  [==TRAIN==]            [==VAL==]
Fold 2:  [====WARMUP====]  [=====TRAIN=====]      [==VAL==]
Fold 3:  [====WARMUP====]  [========TRAIN========] [==VAL==]
Fold 4:  [====WARMUP====]  [===========TRAIN=====] [==VAL==]
```

**Rolling Window**: The training set stays the same size, sliding forward.

```
                     | Training Window (fixed) | Validation
Fold 1:              [==TRAIN==]               [==VAL==]
Fold 2:                    [==TRAIN==]               [==VAL==]
Fold 3:                          [==TRAIN==]               [==VAL==]
Fold 4:                                [==TRAIN==]               [==VAL==]
```

**When to use which?**
- **Expanding window**: When more data is generally better (most ML cases). Later folds benefit from more training data.
- **Rolling window**: When the data distribution shifts over time and old data may hurt (e.g., concept drift).

We use **expanding window** with 4 folds, each with a 7-day validation period (matching the eval horizon).

### 3.2 Our Fold Structure

Here is the concrete layout for our 4-fold expanding-window CV:

| Fold | Warmup (28d) | Training Period | Validation (7d) |
|------|-------------|-----------------|------------------|
| 1 | Apr 20 - May 17 | May 18 - May 28 | May 29 - Jun 4 |
| 2 | Apr 20 - May 17 | May 18 - Jun 4 | Jun 5 - Jun 11 |
| 3 | Apr 20 - May 17 | May 18 - Jun 11 | Jun 12 - Jun 18 |
| 4 | Apr 20 - May 17 | May 18 - Jun 18 | Jun 19 - Jun 25 |

The warmup period is always excluded from training -- it exists only so that lag/rolling features have values.

### 3.3 Models per Fold

For each fold we train **5 LightGBM models**:

| Model | Objective | Weight on Censored | Purpose |
|-------|-----------|-------------------|----------|
| MAE | `regression_l1` | 0.5 | Robust point forecast |
| Huber | `huber` (delta=0.5) | 0.5 | Smooth, outlier-robust |
| Q10 | `quantile` (alpha=0.10) | None | Lower bound of 80% PI |
| Q50 | `quantile` (alpha=0.50) | None | Median forecast |
| Q90 | `quantile` (alpha=0.90) | None | Upper bound of 80% PI |

The **ensemble** = average of (MAE + Huber + Q50) predictions.

In [ ]:
# Define fold structure
available_dates = [d for d in dates if d > warmup_date]
n_avail = len(available_dates)
val_size = 7

fold_specs = []
for fold_i in range(4):
    val_end_idx = n_avail - (3 - fold_i) * val_size
    val_start_idx = val_end_idx - val_size
    fold_specs.append({
        'train_end': available_dates[val_start_idx - 1],
        'val_start': available_dates[val_start_idx],
        'val_end': available_dates[min(val_end_idx - 1, n_avail - 1)],
    })

print(f"Warmup cutoff: {warmup_date.date()}")
print(f"Available dates after warmup: {n_avail}")
print()
for i, fs in enumerate(fold_specs):
    print(f"Fold {i+1}: Train up to {fs['train_end'].date()}, "
          f"Val {fs['val_start'].date()} to {fs['val_end'].date()}")

In [ ]:
# ---- Cross-Validation Loop ----
cv_results = []
cv_model_results = []  # per-model per-fold results
best_models = None
best_val_mae = float('inf')
best_fold = -1

for fold_i, fs in enumerate(fold_specs):
    print(f"\n{'='*60}")
    print(f"  Fold {fold_i+1}/4")
    print(f"{'='*60}")

    tr_mask = (train['dt'] > warmup_date) & (train['dt'] <= fs['train_end'])
    vl_mask = (train['dt'] >= fs['val_start']) & (train['dt'] <= fs['val_end'])

    Xtr, ytr = train.loc[tr_mask, fc].values, train.loc[tr_mask, TARGET].values
    Xvl, yvl = train.loc[vl_mask, fc].values, train.loc[vl_mask, TARGET].values

    # Weights: downweight censored observations
    wtr = np.ones(len(ytr), dtype='float32')
    cens_mask = train.loc[tr_mask, 'cens'].values == 1
    wtr[cens_mask] = 0.5

    print(f"  Train: {len(ytr):,}  |  Val: {len(yvl):,}")

    fold_models = {}

    # --- 1. MAE model ---
    params = {**LGB_BASE, 'objective': 'regression_l1', 'metric': 'mae'}
    dtrain = lgb.Dataset(Xtr, ytr, weight=wtr, feature_name=fc)
    dval = lgb.Dataset(Xvl, yvl, feature_name=fc, reference=dtrain)
    fold_models['mae'] = lgb.train(
        params, dtrain, 800,
        valid_sets=[dval], valid_names=['val'],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )

    # --- 2. Huber model ---
    params = {**LGB_BASE, 'objective': 'huber', 'metric': 'mae', 'huber_delta': 0.5}
    fold_models['huber'] = lgb.train(
        params, dtrain, 800,
        valid_sets=[dval], valid_names=['val'],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )

    # --- 3-5. Quantile models (no weights) ---
    dtr_nw = lgb.Dataset(Xtr, ytr, feature_name=fc)
    dvl_nw = lgb.Dataset(Xvl, yvl, feature_name=fc, reference=dtr_nw)
    for q, qname in [(0.1, 'q10'), (0.5, 'q50'), (0.9, 'q90')]:
        params = {**LGB_BASE, 'objective': 'quantile', 'alpha': q, 'metric': 'quantile'}
        fold_models[qname] = lgb.train(
            params, dtr_nw, 800,
            valid_sets=[dvl_nw], valid_names=['val'],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
        )

    # ---- Evaluate all models on this fold ----
    fold_preds = {}
    for name, mdl in fold_models.items():
        fold_preds[name] = np.clip(mdl.predict(Xvl), 0, None)
    fold_preds['ensemble'] = np.clip(
        (fold_preds['mae'] + fold_preds['huber'] + fold_preds['q50']) / 3, 0, None
    )

    # Per-model MAE for this fold
    for mname in ['mae', 'huber', 'q50', 'ensemble']:
        m_mae = mean_absolute_error(yvl, fold_preds[mname])
        m_rmse = np.sqrt(mean_squared_error(yvl, fold_preds[mname]))
        cv_model_results.append({
            'fold': fold_i + 1, 'model': mname,
            'MAE': m_mae, 'RMSE': m_rmse,
        })

    # Ensemble metrics for fold summary
    ens = fold_preds['ensemble']
    fold_mae = mean_absolute_error(yvl, ens)
    fold_rmse = np.sqrt(mean_squared_error(yvl, ens))
    fold_corr = np.corrcoef(yvl, ens)[0, 1]
    fold_bias = np.mean(ens - yvl)
    fold_cov = np.mean((yvl >= fold_preds['q10']) & (yvl <= fold_preds['q90']))

    cv_results.append({
        'fold': fold_i + 1,
        'MAE': fold_mae, 'RMSE': fold_rmse,
        'Corr': fold_corr, 'Bias': fold_bias,
        'PI_Coverage_80': fold_cov,
        'train_size': len(ytr), 'val_size': len(yvl),
    })

    print(f"  Ensemble: MAE={fold_mae:.4f}  RMSE={fold_rmse:.4f}  "
          f"Corr={fold_corr:.4f}  Bias={fold_bias:+.4f}  PI_cov={fold_cov:.3f}")

    # Keep models from best fold
    if fold_mae < best_val_mae:
        best_val_mae = fold_mae
        best_models = fold_models
        best_fold = fold_i + 1

    del Xtr, ytr, Xvl, yvl, wtr, dtrain, dval, dtr_nw, dvl_nw
    gc.collect()

print(f"\nBest fold: {best_fold} (MAE={best_val_mae:.4f})")

In [ ]:
# ---- CV Summary Table ----
cv_df = pd.DataFrame(cv_results)

print("\n" + "=" * 75)
print("  TEMPORAL CROSS-VALIDATION SUMMARY (Ensemble Model)")
print("=" * 75)
print(cv_df[['fold', 'MAE', 'RMSE', 'Corr', 'Bias', 'PI_Coverage_80']].to_string(index=False))
print("-" * 75)

# Mean +/- std
for metric in ['MAE', 'RMSE', 'Corr', 'Bias', 'PI_Coverage_80']:
    m, s = cv_df[metric].mean(), cv_df[metric].std()
    print(f"  {metric:>15s}: {m:.4f} +/- {s:.4f}")

print("=" * 75)
cv_df.to_csv(os.path.join(NB_OUT, 'nb03_temporal_cv_results.csv'), index=False)
print(f'Saved nb03_temporal_cv_results.csv to {NB_OUT}')

### 3.4 Why CV Is Better Than a Single Split

Compare what we know now vs what we knew from a single split:

| Approach | What you get | What you know |
|----------|-------------|---------------|
| Single split | MAE = X | Almost nothing. Could be lucky or unlucky. |
| 4-fold CV | MAE = X +/- Y | **Confidence interval.** The +/- tells you how stable the model is. |

The standard deviation across folds is your **stability metric**:
- **Small std**: Your model generalizes consistently across different time periods.
- **Large std**: Performance is volatile -- the model may be overfitting to specific temporal patterns.

This is crucial for production deployment: if your model's MAE swings wildly week to week, you cannot trust it for inventory decisions.

---
## Section 4: Feature Importance

Let's examine which features the model relies on most. We use **gain-based importance** from the best fold's MAE model -- this measures how much each feature reduces the loss when used in a split.

In [ ]:
# Feature importance from best fold (gain-based)
imp = pd.DataFrame({
    'feature': fc,
    'importance': best_models['mae'].feature_importance('gain')
})
imp = imp.sort_values('importance', ascending=False).reset_index(drop=True)

# Print top 20
print(f"Top 20 Features (from Fold {best_fold}, gain-based importance):")
print("-" * 50)
for i, row in imp.head(20).iterrows():
    bar = '#' * int(row['importance'] / imp['importance'].max() * 30)
    print(f"  {row['feature']:25s} {row['importance']:>12,.0f}  {bar}")

In [ ]:
# Bar chart of top 20 features
fig, ax = plt.subplots(figsize=(10, 8))
top20 = imp.head(20).iloc[::-1]  # reverse for horizontal bar
ax.barh(top20['feature'], top20['importance'], color=sns.color_palette('viridis', 20))
ax.set_xlabel('Gain-based Importance')
ax.set_title(f'Top 20 Features (Best Fold = Fold {best_fold})')
plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb03_feature_importance_top20.png'), dpi=150, bbox_inches='tight')
plt.show()

### What Do the Top Features Tell Us?

The feature importance reveals the **demand signal hierarchy**:

1. **`dow_prof` / `dow_prof_r`** (day-of-week profile): This dominates because retail demand has strong weekly seasonality. A bakery sells more bread on weekends; an office supply store peaks on weekdays. The model learns each SP's unique weekly pattern.

2. **Stockout history features** (`sor7`, `sor14`, `soh7`, `soh14`): These matter because stockouts distort observed sales. The model uses stockout history to adjust predictions -- if a product stocked out recently, past sales underestimate true demand.

3. **Hierarchy features** (`store_daily_vol`, `cat2_daily_m`, `sp_vs_cat2`): These capture how a product relates to its store and category. A product might be individually noisy, but its category trend is informative. Store volume provides a "rising tide lifts all boats" signal.

4. **Lag and rolling features** (`s_l1`, `s_m7`, `r_m7`): Recent demand is predictive of near-future demand -- the autoregressive signal.

5. **Global stats** (`sp_m`, `pd_m`): The long-run average demand level anchors the predictions.

---
## Section 5: Model Comparison

We trained 5 models per fold. Let's compare how they stack up.

In [ ]:
# Build comparison table: per-model average MAE across folds
cv_model_df = pd.DataFrame(cv_model_results)

model_summary = cv_model_df.groupby('model').agg(
    mean_MAE=('MAE', 'mean'),
    std_MAE=('MAE', 'std'),
    mean_RMSE=('RMSE', 'mean'),
    std_RMSE=('RMSE', 'std'),
).round(4)

# Reorder for clarity
model_order = ['mae', 'huber', 'q50', 'ensemble']
model_summary = model_summary.loc[model_order]

print("\n" + "=" * 65)
print("  MODEL COMPARISON (4-fold CV average)")
print("=" * 65)
for model_name in model_order:
    row = model_summary.loc[model_name]
    print(f"  {model_name:10s}  MAE = {row['mean_MAE']:.4f} +/- {row['std_MAE']:.4f}  "
          f"  RMSE = {row['mean_RMSE']:.4f} +/- {row['std_RMSE']:.4f}")
print("=" * 65)

# Compare single split vs CV
cv_ens_mae = model_summary.loc['ensemble', 'mean_MAE']
cv_ens_std = model_summary.loc['ensemble', 'std_MAE']
print(f"\nSingle-split MAE:        {single_mae:.4f}  (no confidence)")
print(f"CV ensemble MAE:         {cv_ens_mae:.4f} +/- {cv_ens_std:.4f}")
print(f"Difference:              {abs(single_mae - cv_ens_mae):.4f}")
model_summary.to_csv(os.path.join(NB_OUT, 'nb03_model_comparison.csv'))
print(f'Saved nb03_model_comparison.csv to {NB_OUT}')

In [ ]:
# Bar chart: per-model MAE comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Average MAE with error bars
ax = axes[0]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
x_pos = range(len(model_order))
means = [model_summary.loc[m, 'mean_MAE'] for m in model_order]
stds = [model_summary.loc[m, 'std_MAE'] for m in model_order]
ax.bar(x_pos, means, yerr=stds, capsize=5, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels([m.upper() for m in model_order])
ax.set_ylabel('MAE')
ax.set_title('Model Comparison: Average MAE (+/- 1 std)')
# Add single-split reference line
ax.axhline(y=single_mae, color='red', linestyle='--', linewidth=1.5, label=f'Single split MAE = {single_mae:.4f}')
ax.legend()

# Right: Per-fold MAE for each model
ax = axes[1]
for idx, mname in enumerate(model_order):
    fold_maes = cv_model_df[cv_model_df['model'] == mname]['MAE'].values
    ax.plot(range(1, 5), fold_maes, 'o-', color=colors[idx], label=mname.upper(), linewidth=2, markersize=8)
ax.set_xlabel('Fold')
ax.set_ylabel('MAE')
ax.set_title('Per-Fold MAE by Model')
ax.set_xticks([1, 2, 3, 4])
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb03_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

### Model Comparison Insights

**Ensemble vs Individual Models:**
- The ensemble (average of MAE + Huber + Q50) typically **matches or beats** all individual models. This is because each model makes slightly different errors, and averaging cancels out some of those errors. This is the "wisdom of crowds" effect in ML.
- The improvement is usually modest (a few percent), but it comes for free -- no extra data, no extra features.

**MAE vs Huber:**
- MAE (L1 loss) treats all errors equally in absolute terms.
- Huber (delta=0.5) acts like L1 for large errors but L2 for small errors. This makes it more robust to outliers while still giving smooth gradients near zero.
- In practice they are often close, but Huber can be better when data has heavy-tailed demand distributions.

**Quantile Regression (Q10 / Q50 / Q90):**
- These do not minimize MAE -- they minimize the pinball loss at their respective quantiles.
- Q50 minimizes MAE by construction (the median minimizes absolute deviations), so it performs similarly to the MAE model.
- The real value of quantile models is the **prediction interval**: the range [Q10, Q90] should contain 80% of actual values. Check the PI Coverage metric -- is it close to 80%?
- Prediction intervals are critical for inventory optimization: they tell you the range of likely demand, not just the expected value.

---
## Section 6: Key Takeaways

### 1. Why Temporal CV Matters

- A single train/val split gives **one number** -- you do not know its reliability.
- Temporal CV gives you **mean +/- std** -- a confidence interval that tells you how stable your model is.
- In production, models face different weeks with different patterns. CV simulates this and gives you a realistic performance estimate.
- **Always use temporal CV for time-series problems.** Never use random CV (it leaks future information).

### 2. How Many Folds to Use

There is a trade-off:

| Folds | Pros | Cons |
|-------|------|------|
| 2 | Fast | High variance estimate, unreliable |
| 3-4 | Good balance of speed and reliability | Standard choice |
| 5-7 | More robust estimate, tighter confidence | Slower, each fold has less validation data |
| 10+ | Very stable estimate | Very slow, validation windows may overlap |

**Our recommendation**: 3-5 folds for most demand forecasting tasks. The validation window should match your actual forecast horizon (7 days in our case).

### 3. When Ensembles Help vs Do Not

Ensembles help when:
- Individual models make **different types of errors** (diverse models)
- The data has **heterogeneous patterns** (some products benefit from MAE, others from Huber)
- You want **stable production predictions** (ensembles are less volatile than single models)

Ensembles may not help when:
- All models are very similar (e.g., three MAE models with slightly different hyperparameters)
- The data is very clean and one objective clearly dominates
- Latency constraints require a single model

### Summary

| What we learned | Key insight |
|----------------|-------------|
| Single splits are fragile | MAE varies significantly depending on which week you hold out |
| Temporal CV gives confidence | mean +/- std tells you how much to trust your results |
| Feature importance | DOW profile dominates; stockout history and hierarchy features add value |
| Ensembles help | Averaging MAE + Huber + Q50 is cheap and often improves results |
| Quantile models unlock PIs | Q10/Q90 give prediction intervals, essential for inventory decisions |